# 先行研究7：ExpモデルでのLFO

## モデル評価
$$
\text{LFO}
= \sum_{t=t_0}^{T-1} \log{p(x_{t+1}|\vec{x})}
= \sum_{t=t_0}^{T-1} 
\log\left( \int d\theta \ p(x_{t+1} | \theta , \vec{x}) p(\theta | \vec{x})  \right) 
$$
$$
\simeq 
\sum^{T-1}_{t=t_0} \log{ \left(\frac{1}{S} \sum^S_{s=1} 
\left(\frac{1}{R}\sum^{R}_{r=1}\text{Poisson}(x_{t+1} | \exp{(\eta_0^{(s)}} + \alpha^{(s)} y_{t+1}^{(s,r)}))\right)\right) }
$$

$$
y^{(s,r)}_{t+1} |  y^{(s,r)}_t  \sim \mathcal{N} (\theta y^{(s,r)}_t , \sqrt{1-\theta^2}\xi_{t+1})\ , \ \xi_t \sim \mathcal{N}(0,1) 
$$

In [ ]:
# Shared raw data in repository; generated data in this study
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "setup-env.ps1").is_file() and (p / "study_with_me").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from inside paper_study.")
PROJECT_DIR = (
    REPO_ROOT / "Merton_Poisson_Lognormal_Model_Study_and_Negative_Binomial_Extension"
)
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROJECT_DATA_DIR = PROJECT_DIR / "data"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pymc as pm
import pytensor.tensor as pt
import arviz as az
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
#!pip install pymc pytensor
!pip show pymc
!pip show pytensor

numpy: 2.4.2
pandas: 3.0.1
matplotlib: 3.10.8
Name: pymc
Version: 5.28.2
Summary: Probabilistic Programming in Python: Bayesian Modeling and Probabilistic Machine Learning with PyTensor
Home-page: http://github.com/pymc-devs/pymc
Author: 
Author-email: 
License: Apache License, Version 2.0
Location: C:\Users\h20s1\workspace\.venv\Lib\site-packages
Requires: arviz, cachetools, cloudpickle, numpy, pandas, pytensor, rich, scipy, threadpoolctl, typing-extensions
Required-by: 
Name: pytensor
Version: 2.38.2
Summary: Optimizing compiler for evaluating mathematical expressions on CPUs and GPUs.
Home-page: https://github.com/pymc-devs/pytensor
Author: 
Author-email: pymc-devs <pymc.devs@gmail.com>
License-Expression: BSD-3-Clause
Location: C:\Users\h20s1\workspace\.venv\Lib\site-packages
Requires: cons, etuples, filelock, logical-unification, miniKanren, numba, numpy, scipy, setuptools
Required-by: pymc


In [2]:
import sys
from pathlib import Path

# projectルートを追加
sys.path.append(str(PROJECT_DIR))

In [3]:
# make dataframe
from src.data_loader import load_csv_with_datetime_index
from src.preprocessing import normalize_default_counts

df = load_csv_with_datetime_index(
    (RAW_DATA_DIR / "default_counts/M.csv"),
    date_col="Year",
    date_format="%Y",
)

df = normalize_default_counts(df, scale=3000)

from src.data_loader import load_init_params

params = load_init_params()

lambda0_init = params["scalar"]["lambda0_init"]
alpha_init = params["scalar"]["alpha_init"]
theta_init = params["scalar"]["theta_init"]
gamma_init = params["scalar"]["gamma_init"]

lambda0_se = params["scalar"]["lambda0_se"]
alpha_se = params["scalar"]["alpha_se"]
theta_se = params["scalar"]["theta_se"]
gamma_se = params["scalar"]["gamma_se"]

y_init = pd.Series(params["y_init"], index=df.index, name="y_t")

=== CSV loaded ===
shape: (99, 7)
columns: ['Year', 'SG', 'D_SG', 'IG', 'D_IG', 'ALL', 'D_ALL']
=== DatetimeIndex set ===
<class 'pandas.DatetimeIndex'>
DatetimeIndex(['1920-01-01', '1921-01-01', '1922-01-01', '1923-01-01',
               '1924-01-01'],
              dtype='datetime64[us]', name='Year', freq=None)
[normalize_default_counts] created 'SG_norm' from 'D_SG' / 'SG' * 3000
[normalize_default_counts] created 'IG_norm' from 'D_IG' / 'IG' * 3000
[normalize_default_counts] created 'ALL_norm' from 'D_ALL' / 'ALL' * 3000


In [4]:
from src.lfo_poisson_exp import compute_lfo_exp

lfo_exp_total, lfo_exp_df = compute_lfo_exp(
    data=df["ALL_norm"].to_numpy(),
    lambda0_init=lambda0_init,
    alpha_init=alpha_init,
    theta_init=theta_init,
    lambda0_se=lambda0_se,
    alpha_se=alpha_se,
    theta_se=theta_se,
    y_init=y_init,
    prior_scale=1.0,
    t0_grid=list(range(50, len(df["ALL_norm"]) - 1, 1)),
    draws=2000,
    tune=2000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
    verbose=True,
)

print("LFO total =", lfo_exp_total)
print("n_folds   =", len(lfo_exp_df))
print("LFO mean  =", lfo_exp_df["log_pred"].mean())

Initializing NUTS using jitter+adapt_diag...
C:\Users\h20s1\workspace\.venv\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [eta0, alpha, log_l, y]


C:\Users\h20s1\workspace\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 53 seconds.


[1/48] t=50, x_next=8, log_pred=-5.6135


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [eta0, alpha, log_l, y]


...

[47/48] t=96, x_next=49, log_pred=-4.6066


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [eta0, alpha, log_l, y]


C:\Users\h20s1\workspace\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 123 seconds.


[48/48] t=97, x_next=33, log_pred=-4.2617
LFO total = -209.92870912596717
n_folds   = 48
LFO mean  = -4.373514773457649


In [5]:
print("LFO total =", lfo_exp_total)
print("n_folds   =", len(lfo_exp_df))
print("LFO mean  =", lfo_exp_df["log_pred"].mean())

LFO total = -209.92870912596717
n_folds   = 48
LFO mean  = -4.373514773457649
